# Fire impacts simulations

This notebook demonstrates the fire impacts simulation modules.

It is assumed that you have a `FireImpactsProject` populated with all pre-processed data for the catchment.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

from fire_impacts.sim import aggregate_rainfall_data, lumped_daily_rusle, gridded_total_rusle, debris_flow
from fire_impacts import FireImpactsProject

import matplotlib.pyplot as plt


## Load project

In [ ]:
proj = FireImpactsProject('/home/joelrahman/Geospatial/waterra-bushfire/example-project/',exist_ok=True)

In [ ]:
proj.catchments

## Rainfall data

Both the erosion (RUSLE) and debris flow modules rely on subdaily rainfall data. The erosion modules, demonstrated here, relies on 30 minute data, while the debris flow relies on 12 minute data.

Furthermore, it is encouraged that both modules be run stochastically, using multiple rainfall replicates.

The library provides functionality for processing stochastic rainfall data, generated by [pyraingen](https://github.com/crdykman/pyraingen).

We will initially run a short simulation of only three days.

In [ ]:
rain_data_start = '2015-01-01'
rain_data_end = '2015-01-03'

# Pointer to pyraingen generated stochastic rainfall
rainfall_data = '/home/joelrahman/Geospatial/waterra-bushfire/rusle/subdaily_012345.nc'


In [ ]:
aggregate_rainfall_data?

In [ ]:
rainfall_30min = aggregate_rainfall_data(rainfall_data,rain_data_start,rain_data_end)

In [ ]:
rainfall_30min

**Note:** The pyraingen data includes 10 stochastic replicates. In the following examples, we will only use one replicate.


## Erosion - Daily simulation

Erosion processed are modelled cell-by-cell, timestep-by-timestep.

This can result in a large amount of output that in most cases is not directly usable.

The library currently provides two alternative function calls, depending on the nature of outputs required:

* `lumped_daily_rusle`: For modelling erosion processes and receiving daily outputs lumped to the subcatchment scale, and
* `gridded_total_rusle`: For modelling erosion processes and receiving gridded outputs summing the erosion over the simulation period.

Both of these functions rely on a low level function which can be used to provide customise outputs


First, lets run the lumped daily function:

In [ ]:
lumped_daily_rusle?

In [ ]:
lumped_daily_rusle(proj,rainfall_30min[9],'DR_Primary_Catchment_Thomson')

## Erosion - Gridded simulation

This time, we will run for a longer time period (2 years) and retrieve a single map of predicted erosion over this time period

In [ ]:
rain_data_start = '2015-01-01'
rain_data_end = '2016-12-31'
rainfall_30min = aggregate_rainfall_data(rainfall_data,rain_data_start,rain_data_end)

In [ ]:
gridded_total_rusle?

In [ ]:
%time eroded, delivered, transform = gridded_total_rusle(proj,rainfall_30min[9],'DR_Primary_Catchment_Thomson')

In [ ]:
plt.imshow(eroded)
plt.colorbar()

In [ ]:
plt.imshow(delivered)
plt.colorbar()

## Debris Flow

Debris flow operates on headwaters only and uses 12 minute rainfall data.


In [ ]:
rain_data_start = '2015-01-01'
rain_data_end = '2016-12-31'
rainfall = aggregate_rainfall_data(rainfall_data,rain_data_start,rain_data_end,time_res='12min')
rainfall

In [ ]:
results = debris_flow(proj,rainfall[9])

In [ ]:
results['DR_Primary_Catchment_Thomson']